# ⚡ Tier-1: Smart Grid Terminal & Circuit Breaker Fault Prediction
### Siemens IOT-01 — Predictive Failure Detection for Smart Grid Substations
**Author:** Sovereign Predictive Maintenance Team | **Target Model:** High-Frequency Terminal Telemetry Engine  
**Dataset Reference:** `zara2099/smart-grid-terminal-fault-dataset`

---

## 🎯 Executive Overview & Equipment Context
In a high-voltage electrical substation, terminal failure can trigger instantaneous breaker lockout, arching, or catastrophic phase flashover:
* **Monitored Assets:**
  - **High-Voltage SF6 Circuit Breakers** (Siemens 3AP1 / 3AV1 145 kV)
  - **Gas-Insulated Switchgear (GIS) Terminal Bays** (Siemens 8DN8)
  - **Feeder Protection IEDs & RTUs** (Siemens SIPROTEC 5 / SICAM A8000)
  - **Motor-Operated Disconnector Terminals**
* **Primary Failure Modes:** Terminal contact degradation, phase current unbalance, vibration-induced mechanical stiction, voltage sags/swells, and thermal runaway under overload.

---

## 📊 Notebook Workflow
1. **Environment Setup & Kaggle Ingestion Pipeline**
2. **Exploratory Data Analysis (EDA) on 15 Terminal Parameters**
3. **Domain Feature Engineering (Power Systems Physics)**
4. **Model Architecture: LightGBM vs. XGBoost vs. Random Forest**
5. **Rigorous Evaluation (ROC-AUC, Precision, Recall, F1, PR-Curves)**
6. **Explainability & Sensor Attribution (TreeSHAP Waterfall & Feature Rankings)**
7. **Artifact Export for Substation Production API**

In [ ]:
# 1. Environment Setup & Core Libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, accuracy_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

import xgboost as xgb
import lightgbm as lgb
import shap

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Environment initialized successfully.")

## 2. Data Ingestion & Schema Alignment
The dataset contains 15 core parameters representing operational states of smart grid terminals.  
If the local file `data/terminal_faults.csv` is present, it is loaded; otherwise, an IEEE-compliant synthetic simulator generates an identical schema distribution.

In [ ]:
# 2. Data Loading with Automatic Schema Verification
DATASET_PATH = "data/terminal_faults.csv"
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

EXPECTED_COLUMNS = [
    "Voltage(V)", "Current(A)", "Temperature(C)", "Frequency(Hz)",
    "Power_Factor", "Vibration_Level", "Device_Age(Years)", "Terminal_Efficiency(%)",
    "Load_Variation(%)", "Signal_Strength(dB)", "Apparent_Power(kVA)", "Real_Power(kW)",
    "Energy_Consumed(kWh)", "Anomaly_Score", "Fault_Status"
]

def load_or_generate_terminal_dataset(path: str, n_samples: int = 1500) -> pd.DataFrame:
    if os.path.exists(path):
        print(f"Loading real dataset from: {path}")
        df = pd.read_csv(path)
        return df
    
    print(f"Dataset not found at '{path}'. Generating IEEE-aligned simulation dataset ({n_samples} samples)...")
    np.random.seed(42)
    
    # Normal operational baseline
    voltage = np.random.normal(230.0, 4.5, n_samples)
    current = np.random.gamma(shape=5.0, scale=8.0, size=n_samples)
    temperature = np.random.normal(45.0, 8.0, n_samples)
    frequency = np.random.normal(50.0, 0.15, n_samples)
    power_factor = np.clip(np.random.normal(0.92, 0.05, n_samples), 0.65, 0.99)
    vibration = np.random.exponential(scale=1.5, size=n_samples)
    device_age = np.random.uniform(1.0, 15.0, n_samples)
    efficiency = np.clip(100.0 - (temperature * 0.2 + device_age * 0.8 + np.random.normal(0, 2, n_samples)), 60, 99.5)
    load_variation = np.random.uniform(5.0, 35.0, n_samples)
    signal_strength = np.random.normal(-65.0, 10.0, n_samples)
    apparent_power = (voltage * current) / 1000.0
    real_power = apparent_power * power_factor
    energy_consumed = real_power * np.random.uniform(10.0, 24.0, n_samples)
    
    # Calculate baseline latent risk score
    z_temp = (temperature - 45.0) / 8.0
    z_vib = (vibration - 1.5) / 1.5
    z_load = (load_variation - 20.0) / 10.0
    z_volt = np.abs(voltage - 230.0) / 4.5
    
    latent_risk = 0.35 * z_temp + 0.30 * z_vib + 0.20 * z_load + 0.15 * z_volt + (device_age / 20.0)
    anomaly_score = 1.0 / (1.0 + np.exp(-latent_risk))
    
    # Fault injection (~18% fault rate representing rare catastrophic trips)
    fault_prob = anomaly_score ** 2.2
    fault_status = (np.random.rand(n_samples) < fault_prob).astype(int)
    
    # Emphasize fault signatures for anomalous samples
    temperature[fault_status == 1] += np.random.uniform(20, 50, sum(fault_status == 1))
    vibration[fault_status == 1] += np.random.uniform(3.5, 8.0, sum(fault_status == 1))
    voltage[fault_status == 1] += np.random.choice([-1, 1], sum(fault_status == 1)) * np.random.uniform(15, 40, sum(fault_status == 1))
    
    df = pd.DataFrame({
        "Voltage(V)": np.round(voltage, 2),
        "Current(A)": np.round(current, 2),
        "Temperature(C)": np.round(temperature, 2),
        "Frequency(Hz)": np.round(frequency, 3),
        "Power_Factor": np.round(power_factor, 3),
        "Vibration_Level": np.round(vibration, 3),
        "Device_Age(Years)": np.round(device_age, 1),
        "Terminal_Efficiency(%)": np.round(efficiency, 2),
        "Load_Variation(%)": np.round(load_variation, 2),
        "Signal_Strength(dB)": np.round(signal_strength, 1),
        "Apparent_Power(kVA)": np.round(apparent_power, 3),
        "Real_Power(kW)": np.round(real_power, 3),
        "Energy_Consumed(kWh)": np.round(energy_consumed, 2),
        "Anomaly_Score": np.round(anomaly_score, 4),
        "Fault_Status": fault_status
    })
    
    df.to_csv(path, index=False)
    print(f"Dataset cached at: {path}")
    return df

df_terminal = load_or_generate_terminal_dataset(DATASET_PATH)
df_terminal.head()

## 3. Exploratory Data Analysis & Statistical Sanity Checks
Inspect the class balance and distributions of key sensors driving breaker trips.

In [ ]:
print(f"Dataset Shape: {df_terminal.shape}")
print(f"Class Distribution: \n{df_terminal['Fault_Status'].value_counts(normalize=True)}")

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
sns.boxplot(data=df_terminal, x='Fault_Status', y='Temperature(C)', ax=axes[0], palette='Set2')
axes[0].set_title('Terminal Temperature vs. Fault')

sns.boxplot(data=df_terminal, x='Fault_Status', y='Vibration_Level', ax=axes[1], palette='Set2')
axes[1].set_title('Mechanical Vibration vs. Fault')

sns.boxplot(data=df_terminal, x='Fault_Status', y='Load_Variation(%)', ax=axes[2], palette='Set2')
axes[2].set_title('Load Variation vs. Fault')

sns.boxplot(data=df_terminal, x='Fault_Status', y='Voltage(V)', ax=axes[3], palette='Set2')
axes[3].set_title('Voltage Fluctuations vs. Fault')
plt.tight_layout()
plt.show()

## 4. Power Systems Feature Engineering
We calculate physics-based domain features:
1. **Thermal Stress Index:** $I^2 R$ heat proxy: $T \times (I / I_{\text{mean}})^2$
2. **Voltage Fluctuation Severity:** $|V - 230.0| / 230.0$
3. **Frequency Deviation:** $|f - 50.0|$
4. **Apparent vs Active Power Residual:** $|S - P / \text{PF}|$
5. **Dynamic Impedance Proxy:** $Z = V / (I + 1e-4)$

In [ ]:
def engineer_terminal_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    
    nominal_v = 230.0
    nominal_f = 50.0
    mean_i = data['Current(A)'].mean()
    
    # 1. Thermal Overload Factor
    data['Thermal_Stress_Index'] = data['Temperature(C)'] * ((data['Current(A)'] / (mean_i + 1e-5)) ** 2)
    
    # 2. Voltage Deviation Ratio
    data['Voltage_Deviation_Ratio'] = np.abs(data['Voltage(V)'] - nominal_v) / nominal_v
    
    # 3. Frequency Deviation Severity
    data['Frequency_Deviation'] = np.abs(data['Frequency(Hz)'] - nominal_f)
    
    # 4. Apparent vs Active Discrepancy
    data['Power_Factor_Loss'] = 1.0 - data['Power_Factor']
    
    # 5. Dynamic Impedance Proxy
    data['Impedance_Proxy'] = data['Voltage(V)'] / (data['Current(A)'] + 0.01)
    
    # 6. Combined Mechanical-Thermal Shock
    data['Mech_Thermal_Shock'] = data['Vibration_Level'] * data['Temperature(C)'] / 100.0
    
    return data

df_engineered = engineer_terminal_features(df_terminal)
print(f"Features after engineering: {df_engineered.shape[1]}")

## 5. Model Training & Comparison (LightGBM vs. XGBoost vs. Baseline)
We split data into 80% train and 20% test using Stratified Splitting to preserve class imbalance.

In [ ]:
X = df_engineered.drop(columns=['Fault_Status'])
y = df_engineered['Fault_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

# Calculate scale_pos_weight for class imbalance handling
pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
print(f"Training Samples: {len(X_train)} | Test Samples: {len(X_test)} | Pos Weight: {pos_weight:.2f}")

# Train Models
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "RandomForest": RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, class_weight='balanced'),
    "XGBoost": xgb.XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, scale_pos_weight=pos_weight, random_state=42, eval_metric='logloss'),
    "LightGBM": lgb.LGBMClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, scale_pos_weight=pos_weight, random_state=42, verbose=-1)
}

results = {}
for name, model in models.items():
    if name == "LogisticRegression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    results[name] = {"Accuracy": acc, "F1-Score": f1, "ROC-AUC": roc, "Model": model}
    print(f"[{name:18s}] Acc: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {roc:.4f}")

df_results = pd.DataFrame(results).T[['Accuracy', 'F1-Score', 'ROC-AUC']]
df_results

## 6. Champion Model Evaluation & Diagnostic Curves
Let's select LightGBM/XGBoost and analyze the ROC Curve and Precision-Recall tradeoffs.

In [ ]:
champion_name = "LightGBM"
champion_model = results[champion_name]["Model"]
y_pred_champ = champion_model.predict(X_test)
y_prob_champ = champion_model.predict_proba(X_test)[:, 1]

print("=== CHAMPION MODEL CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred_champ, target_names=["Nominal (Healthy)", "Fault (Trip)"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_champ)
ax1.plot(fpr, tpr, label=f'{champion_name} (AUC = {roc_auc_score(y_test, y_prob_champ):.3f})', color='#0284c7', lw=2)
ax1.plot([0, 1], [0, 1], 'k--', lw=1)
ax1.set_title('Receiver Operating Characteristic (ROC)')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.legend(loc='lower right')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_champ)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2, cbar=False,
            xticklabels=["Nominal", "Fault"], yticklabels=["Nominal", "Fault"])
ax2.set_title(f'{champion_name} Confusion Matrix')
ax2.set_xlabel('Predicted Status')
ax2.set_ylabel('Actual Ground Truth')

plt.tight_layout()
plt.show()

## 7. Explainability & Sensor Attribution (TreeSHAP)
**This fulfills Deliverable 2 of the Siemens Problem Statement:**
*"Flags the specific sensor readings driving the prediction."*

Using TreeSHAP, we compute exact Shapley feature attributions to answer **why** a breaker tripped.

In [ ]:
# Compute SHAP Values
explainer = shap.TreeExplainer(champion_model)
shap_values = explainer.shap_values(X_test)

# If LightGBM returns list of 2 arrays, pick positive class [1]
if isinstance(shap_values, list):
    shap_vals_fault = shap_values[1]
else:
    shap_vals_fault = shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals_fault, X_test, plot_type="bar", show=False)
plt.title("Substation Terminal Fault: Global Sensor Driving Attribution (SHAP)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Local Sample Explanation: Diagnostic Inspection of a Tripped Breaker
fault_indices = np.where((y_test.values == 1) & (y_pred_champ == 1))[0]
sample_idx = fault_indices[0] if len(fault_indices) > 0 else 0

sample_data = X_test.iloc[[sample_idx]]
sample_prob = champion_model.predict_proba(sample_data)[0, 1]

print(f"Sample #{sample_idx} Failure Probability: {sample_prob * 100:.2f}%")
print("\nTop Driving Sensor Anomalies for this Trip:")

# Extract top SHAP contributors
if isinstance(shap_values, list):
    single_shap = shap_values[1][sample_idx]
else:
    single_shap = shap_values[sample_idx]

top_features = pd.Series(single_shap, index=X_test.columns).sort_values(ascending=False).head(5)
for feat, val in top_features.items():
    actual_val = sample_data[feat].values[0]
    print(f"  * {feat:<26s} = {actual_val:8.2f} (SHAP impact: +{val:+.4f})")

## 8. Exporting Artifacts for Substation API
We serialize the trained LightGBM model, feature list, and scaler for real-time deployment in `src/api.py`.

In [ ]:
artifacts = {
    "model": champion_model,
    "feature_names": list(X.columns),
    "scaler": scaler,
    "threshold": 0.50,
    "version": "1.0.0-siemens-terminal"
}

joblib.dump(artifacts, "models/terminal_fault_model.joblib")
print("Saved artifacts to 'models/terminal_fault_model.joblib' successfully!")